# Prefix GCN — Sliding-Window Classification on Helpdesk

This notebook demonstrates **prefix-based next-event prediction** using a
Graph Convolutional Network (GCN) on the Helpdesk event log.

## Model overview

| Component | Description |
|-----------|-------------|
| Architecture | Dual-path GCN (embedding + features) with `global_mean_pool` for graph-level readout |
| Prefix approach | Each case is split into sliding-window **sub-sequences** of fixed `prefix_size` |
| Sequence features | Case-level features are concatenated to the pooled graph embedding via a feed-forward head |
| Task | Predict the **event immediately following** each prefix |

Unlike the GAT models (which predict next-event at every node), PrefixGCN produces
a single **graph-level classification** per prefix subsequence.

### Notebook outline
1. Load & Explore the Dataset
2. Train the Model
3. Evaluate with Top-k Metrics
4. Summary

In [1]:
from pathlib import Path
import sys, warnings

root = Path.cwd().resolve()
src = root / "src" if (root / "src").exists() else root.parent / "src"
sys.path.insert(0, str(src))
warnings.filterwarnings("ignore", category=FutureWarning)

## 1. Load & Explore the Dataset

In [2]:
import pandas as pd

data_dir = Path.cwd() / "data" if (Path.cwd() / "data").exists() else Path.cwd() / "examples" / "data"
df = pd.read_csv(data_dir / "Helpdesk.csv")

df = df.rename(columns={
    "case:concept:name": "case_id",
    "concept:name":      "event",
    "time:timestamp":    "time",
    "lifecycle:transition": "status",
    "case:variant-index":   "variant_index",
})
df = df.dropna(subset=["case_id", "event", "time"]).copy()
df["time"] = pd.to_datetime(df["time"], utc=True)
df = df[df.groupby("case_id")["case_id"].transform("size") >= 2].reset_index(drop=True)

# Show the distribution of sequence lengths — important for choosing prefix_size
lengths = df.groupby("case_id").size()
print(f"Cases: {len(lengths)}  |  Events: {len(df)}")
print(f"Sequence lengths — min: {lengths.min()}, median: {lengths.median():.0f}, "
      f"max: {lengths.max()}, mean: {lengths.mean():.1f}")
df.head()

Cases: 4580  |  Events: 21348
Sequence lengths — min: 2, median: 4, max: 15, mean: 4.7


,event,status,org:resource,time,Activity,Resource,case_id,case:variant,variant_index,case:creator
0,Assign seriousness,complete,Value 2,2010-01-13 08:40:25+00:00,Assign seriousness,Value 2,Case3608,Variant 33,33,Fluxicon Disco
1,Take in charge ticket,complete,Value 2,2010-01-29 08:52:27+00:00,Take in charge ticket,Value 2,Case3608,Variant 33,33,Fluxicon Disco
2,Resolve ticket,complete,Value 2,2010-01-29 08:52:34+00:00,Resolve ticket,Value 2,Case3608,Variant 33,33,Fluxicon Disco
3,Closed,complete,Value 5,2010-02-13 08:52:48+00:00,Closed,Value 5,Case3608,Variant 33,33,Fluxicon Disco
4,Closed,complete,Value 5,2010-02-13 08:52:48+00:00,Closed,Value 5,Case3608,Variant 33,33,Fluxicon Disco


## 2. Train the Model

Key parameter: **`prefix_size`** determines the length of the sliding window.
Cases shorter than `prefix_size` are automatically excluded.

The recipe handles:
- Generating all valid (prefix, label) pairs via a sliding window
- Building one graph per prefix
- Train/test split with optional stratification

In [3]:
from timegnn import prefix_gcn

result = prefix_gcn(
    df,
    case_col="case_id",
    event_col="event",
    time_col="time",
    cat_event=["event"],
    num_event=[],
    cat_seq=[],
    num_seq=["variant_index"],
    prefix_size=5,
    num_epochs=3,
    batch_size=16,
    stratify=False,
)

model = result["model"]
pd.DataFrame(result["history"])

,epoch,train_loss,train_acc,test_loss,test_acc
0,1,0.773422,0.768547,0.513118,0.864198
1,2,0.478028,0.869397,0.481742,0.859568
2,3,0.438460,0.874807,0.485752,0.851852


### Optional: full PrefixGCN advanced configuration

This example includes all major controls:
- depth (`num_layers`)
- regularization (`dropout`, `use_batch_norm`)
- hidden activation (`activation`)
- early stopping (`patience`, `delta`)

In [ ]:
from timegnn import PrefixGCNConfig, prefix_gcn

cfg = PrefixGCNConfig(
    prefix_size=10,
    num_layers=3,
    dropout=0.20,
    use_batch_norm=True,
    activation="elu",
    num_epochs=30,
    patience=5,
    delta=1e-4,
    batch_size=32,
)

result_cfg = prefix_gcn(
    df,
    case_col="case_id",
    event_col="event",
    time_col="time",
    cat_event=["event"],
    num_event=[],
    cat_seq=[],
    num_seq=["variant_index"],
    config=cfg,
)

pd.DataFrame(result_cfg["history"]).tail()

,epoch,train_loss,train_acc,test_loss,test_acc
15,16,1.273140,0.448980,1.355859,0.461538
16,17,1.257857,0.448980,1.346023,0.461538
17,18,1.226069,0.530612,1.334508,0.538462
18,19,1.206305,0.551020,1.308313,0.615385
19,20,1.161753,0.510204,1.298007,0.538462


## 3. Evaluate with Top-k Metrics

Since PrefixGCN produces one prediction per prefix (not per-node),
sequence-level metrics like BLEU/DLS are not applicable.
We use **top-k accuracy** instead.

In [5]:
import torch
from torch.utils.data import DataLoader
from timegnn import top_k_accuracy
from timegnn.data import PrefixGCNTransformer
from timegnn.data.pyg import PrefixDataset, custom_collate_prefix

PREFIX_SIZE = 5

# Keep only cases long enough for the prefix window
df_prefix = df[df.groupby("case_id")["case_id"].transform("size") >= PREFIX_SIZE].reset_index(drop=True)

transformer = PrefixGCNTransformer(
    case_col="case_id", event_col="event", time_col="time",
    prefix_size=PREFIX_SIZE,
    cat_event=["event"], num_event=[],
    cat_seq=[], num_seq=["variant_index"],
)
transformed = transformer.transform(df_prefix)

dataset = PrefixDataset(transformed.event_features, transformed.sequence_features, transformed.labels)
loader  = DataLoader(dataset, batch_size=16, shuffle=False, collate_fn=custom_collate_prefix)

device = "cuda" if torch.cuda.is_available() else "cpu"
model  = model.to(device).eval()

all_outputs, all_labels = [], []
with torch.no_grad():
    for batch_event, batch_seq, batch_labels in loader:
        out = model(batch_event.to(device), batch_seq.to(device))
        all_outputs.append(out.cpu())
        all_labels.append(batch_labels.cpu())

outputs = torch.cat(all_outputs)
labels  = torch.cat(all_labels)

print(f"Top-1 accuracy: {top_k_accuracy(outputs, labels, k=1):.4f}")
print(f"Top-3 accuracy: {top_k_accuracy(outputs, labels, k=3):.4f}")
print(f"Top-5 accuracy: {top_k_accuracy(outputs, labels, k=5):.4f}")
print(f"\nTotal prefix samples evaluated: {len(labels)}")

Top-1 accuracy: 0.8582
Top-3 accuracy: 0.9756
Top-5 accuracy: 0.9947

Total prefix samples evaluated: 3238


## 4. Summary

The **Prefix GCN** model provides a fixed-window view of the process.

| Advantage | Limitation |
|-----------|------------|
| Controlled input size → stable training | Cases shorter than `prefix_size` are excluded |
| Includes sequence-level features in the classifier head | No attention mechanism for interpretability |
| Naturally suited for graph-level tasks (outcome, next-event) | Multiple samples per case → larger dataset |

For full-sequence next-event prediction, consider the GAT-based models instead.